## Concept 5 — RAG Fusion

### What is it?
A single user query is just one way to ask a question. RAG Fusion generates **multiple rephrased versions** of the query, retrieves docs for each, then merges all results using **Reciprocal Rank Fusion (RRF)**.

### Pipeline
```
Query
  → LLM generates N query variations
  → Retrieve docs for EACH variation (parallel)
  → Merge all results with RRF scoring
  → Top docs → LLM → Answer
```

### What is RRF?
Reciprocal Rank Fusion is a formula that merges multiple ranked lists.
A doc that appears in top positions across *multiple* query results gets a high final score.
Formula: `score += 1 / (rank + k)` for each list it appears in.

### Why it's better than single query
One query may miss docs that match a slightly different phrasing.
4 query variations cast a wider net — together they find more relevant docs.

### Limitation (what Concept 6 fixes)
Queries and answer-docs live in different embedding spaces — short questions don't match well with long detailed answers.
→ Concept 6 (HyDE) fixes this by embedding a hypothetical answer instead of the question.

In [ ]:
import sys
sys.path.insert(0, '.')

from RAGutils import setup, format_docs, TEST_QUERIES, run_queries

chunks, vectorstore, embeddings, llm, prompt = setup()

### Step 1 — Generate Multiple Query Variations
Ask the LLM to rephrase the query in different ways, covering different angles.

In [ ]:
def generate_queries(original_query, n=4):
    gen_prompt = (
        f"Generate {n} different search query variations for the question below.\n"
        f"Each variation should cover a different angle or use different words.\n"
        f"Return ONLY the queries, one per line, no numbering, no extra text.\n\n"
        f"Question: {original_query}"
    )
    response = llm.invoke(gen_prompt).content.strip()
    queries  = [q.strip() for q in response.split('\n') if q.strip()]
    return queries[:n]

# Test
query = TEST_QUERIES[0]
variations = generate_queries(query)
print(f"Original: {query}\n")
print("Generated variations:")
for i, v in enumerate(variations, 1):
    print(f"  {i}. {v}")

### Step 2 — Reciprocal Rank Fusion (RRF)
Merge multiple ranked doc lists into one. Docs appearing high across multiple lists score highest.

In [ ]:
def reciprocal_rank_fusion(results_list, k=60):
    """
    results_list: list of lists of Documents (one per query)
    k: smoothing constant (60 is standard)
    Returns: merged and re-ranked list of Documents
    """
    scores = {}  # content → {score, doc}

    for results in results_list:
        for rank, doc in enumerate(results):
            key = doc.page_content
            if key not in scores:
                scores[key] = {"score": 0.0, "doc": doc}
            # RRF formula: 1 / (rank + k)
            scores[key]["score"] += 1.0 / (rank + k)

    sorted_docs = sorted(scores.values(), key=lambda x: x["score"], reverse=True)
    return [item["doc"] for item in sorted_docs]

### Step 3 — See RRF in Action
Retrieve for each query variation and observe which docs rise to the top.

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

query      = TEST_QUERIES[0]
variations = generate_queries(query, n=4)

all_results = []
for v in variations:
    docs = retriever.invoke(v)
    all_results.append(docs)
    print(f"Query '{v[:50]}...' → {len(docs)} docs")

fused_docs = reciprocal_rank_fusion(all_results)
print(f"\nAfter RRF fusion: {len(fused_docs)} unique docs, ranked by cross-query relevance")

### Step 4 — Full RAG Fusion Function

In [ ]:
def rag_fusion(query, n_queries=4, top_k=5):
    # Generate query variations
    variations  = generate_queries(query, n=n_queries)

    # Retrieve for each variation
    all_results = [retriever.invoke(v) for v in variations]

    # Merge with RRF and keep top_k
    fused_docs = reciprocal_rank_fusion(all_results)[:top_k]

    # Answer
    context  = format_docs(fused_docs)
    response = llm.invoke(prompt.format_messages(context=context, question=query))
    return response.content

### Test — Standard Queries

In [ ]:
run_queries(rag_fusion, label="Concept 5 — RAG Fusion")